<a href="https://colab.research.google.com/github/manthansingh26/FLY_Manthan/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip -q install duckdb huggingface_hub

In [11]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Connected to the FlyRank warehouse.")

Connected to the FlyRank warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The source data grain is one pseudonymized content page for one client on one report date.

In other words, one source row = `report_date × client_hash_id × content_hash_id`.

For this Week 3 contract, I use the mid-panel month from 2026-03-01 to 2026-03-31. I chose March 2026 because it is not the final month of the dataset.

My final decision frame will later aggregate these daily records so that one row represents one content page. The output supports ranking pages for content review.

In [12]:
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,client_count,content_count,first_report_date,last_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
I will build five features from the first 15 days of March 2026: impressions, clicks, average Google Search position, days with impressions, and click-through rate.

These features are knowable at the decision moment, March 16, 2026.

### Label / proxy
My proxy label is `is_declining_label`.

It equals 1 when a content page's impressions in the second half of March are at least 20% lower than its impressions in the first half. Otherwise it equals 0.

The second-half impressions define the outcome, so they are never used as features.

### Context
`client_hash_id` and `content_hash_id` are pseudonymized IDs. I use them only for grouping, joining, and later validation, not as model features. `report_date` defines the time window.

### Excluded
I exclude client IDs and content IDs as features because IDs can cause memorization instead of learning a general pattern.

I exclude future-period metrics used to create the label because that would cause data leakage.

I do not use client names, URLs, raw queries, or private information.

In [13]:
con.sql(f"""
    DESCRIBE SELECT *
    FROM {TABLES["fact_daily"]}
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verify three facts with queries:

1. The March 2026 row count and date window are shown in Section 1.
2. The query below checks whether the stated daily grain has duplicates.
3. The final query checks GA4 availability and missing average-position values.

In [14]:
con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS duplicate_count
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,duplicate_count


In [15]:
con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        SUM(
            CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END
        ) AS ga4_available_rows,
        ROUND(
            100.0 * AVG(
                CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END
            ),
            2
        ) AS ga4_available_percent,
        SUM(
            CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END
        ) AS missing_position_rows
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,ga4_available_rows,ga4_available_percent,missing_position_rows
0,9841378,413966.0,4.21,6230317.0


### Five-feature frame

I create one row per content page.

The five features use only March 1–15 and are knowable at the decision moment, March 16.

- `impressions_prev15`: observed impressions before the decision moment.
- `clicks_prev15`: observed clicks before the decision moment.
- `avg_position_prev15`: observed average search position before the decision moment.
- `days_with_impressions_prev15`: observed number of active impression days before the decision moment.
- `ctr_prev15`: observed click-through rate before the decision moment.

The label uses March 16–31 as the later observed outcome. This keeps the feature window separate from the outcome window.

In [16]:
feature_frame = con.sql(f"""
    WITH previous_period AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_prev15,
            SUM(gsc_clicks) AS clicks_prev15,
            AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev15,
            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0 THEN report_date
                END
            ) AS days_with_impressions_prev15
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-03-16'
        GROUP BY 1, 2
    ),
    outcome_period AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_next15
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= DATE '2026-03-16'
          AND report_date < DATE '2026-04-01'
        GROUP BY 1, 2
    )
    SELECT
        p.*,
        100.0 * p.clicks_prev15
            / NULLIF(p.impressions_prev15, 0) AS ctr_prev15,
        o.impressions_next15,
        CASE
            WHEN o.impressions_next15 < 0.8 * p.impressions_prev15
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM previous_period p
    INNER JOIN outcome_period o
        USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prev15 >= 100
""").df()

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_prev15,clicks_prev15,avg_position_prev15,days_with_impressions_prev15,ctr_prev15,impressions_next15,is_declining_label
0,client_3197e6291363b4db,content_bdaa382ed57031a3,132.0,0.0,6.178937,15,0.000000,124.0,0
1,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,879.0,1.0,8.027743,15,0.113766,655.0,1
2,client_ff644d8251367cbb,content_cf3aa48656798425,747.0,0.0,14.061160,15,0.000000,455.0,1
3,client_ff644d8251367cbb,content_80218760d6f2309f,2027.0,1.0,5.605966,15,0.049334,1551.0,1
4,client_ff644d8251367cbb,content_64be55d2ba6eb0e1,2110.0,12.0,1.943981,15,0.568720,2136.0,0
5,client_ff644d8251367cbb,content_d9ce64a0799888cc,494.0,4.0,3.951108,15,0.809717,472.0,0
6,client_ff644d8251367cbb,content_5d51db7f4ab44d89,1485.0,12.0,4.716733,15,0.808081,2310.0,0
7,client_ff644d8251367cbb,content_d47f539ab9a634a1,280.0,3.0,11.184957,15,1.071429,220.0,1
8,client_ff644d8251367cbb,content_6469be23f8a83853,1017.0,4.0,8.316565,15,0.393314,1080.0,0
9,client_ff644d8251367cbb,content_8344d96e0e8ad6d8,1509.0,4.0,3.547408,15,0.265076,2349.0,0


### Deliberate leakage check

I deliberately add one label-derived column to show why leakage is dangerous.

The copied label produces an artificially near-perfect score because it reveals the answer to the model. I remove that column immediately and keep only the honest score from the five safe features.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

safe_feature_cols = [
    "impressions_prev15",
    "clicks_prev15",
    "avg_position_prev15",
    "days_with_impressions_prev15",
    "ctr_prev15",
]

model_frame = feature_frame.dropna(subset=safe_feature_cols).copy()

train_idx, test_idx = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame["is_declining_label"],
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
).fit(
    model_frame.loc[train_idx, safe_feature_cols],
    model_frame.loc[train_idx, "is_declining_label"],
)

honest_accuracy = accuracy_score(
    model_frame.loc[test_idx, "is_declining_label"],
    honest_model.predict(model_frame.loc[test_idx, safe_feature_cols]),
)

# Deliberate mistake: this column is copied directly from the label.
leaky_frame = model_frame.copy()
leaky_frame["leaky_label_copy"] = leaky_frame["is_declining_label"]

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
).fit(
    leaky_frame.loc[train_idx, safe_feature_cols + ["leaky_label_copy"]],
    leaky_frame.loc[train_idx, "is_declining_label"],
)

leaky_accuracy = accuracy_score(
    leaky_frame.loc[test_idx, "is_declining_label"],
    leaky_model.predict(
        leaky_frame.loc[test_idx, safe_feature_cols + ["leaky_label_copy"]]
    ),
)

print(f"Honest accuracy using five safe features: {honest_accuracy:.3f}")
print(f"Leaky accuracy using a copied label: {leaky_accuracy:.3f}")

# Remove the leaking column. It must never be used again.
leaky_frame = leaky_frame.drop(columns=["leaky_label_copy"])

Honest accuracy using five safe features: 0.718
Leaky accuracy using a copied label: 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot prove that refreshing a page causes traffic recovery. It only measures observed search-performance patterns.

Client histories are unbalanced: different clients have different amounts of historical data. Therefore, a result from one month may not represent every client.

GA4 is available for only 4.21% of March 2026 rows in this slice. GA4 values before availability can be zero-filled, so zero does not always mean zero engagement. I therefore exclude GA4 metrics from this feature frame.

Average search position is missing for many rows, so the feature frame treats unavailable position values carefully rather than assuming they mean a strong ranking.

The `is_declining_label` is a defined proxy based on later observed impressions. It does not prove a causal business outcome or explain Google's algorithm.

The output is directional decision-support: it helps rank pages for content review, not make guaranteed refresh claims.

In [18]:
print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    "Observed declining-label rate: "
    f"{feature_frame['is_declining_label'].mean():.3f}"
)

feature_frame[[
    "impressions_prev15",
    "clicks_prev15",
    "avg_position_prev15",
    "days_with_impressions_prev15",
    "ctr_prev15",
]].isna().sum()

Feature-frame rows: 77,540
Observed declining-label rate: 0.285


,0
impressions_prev15,0
clicks_prev15,0
avg_position_prev15,0
days_with_impressions_prev15,0
ctr_prev15,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.